<a href="https://colab.research.google.com/github/sami01979/fer_pattern/blob/main/mobilenetv3L.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())

2.11.0+cu130 True


In [2]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive | grep -i zip

Mounted at /content/drive


In [4]:
!mkdir -p /content/raf
!unzip -q /content/drive/MyDrive/DATASET.zip -d /content/raf
!find /content/raf -maxdepth 3 -type d | sort

/content/raf
/content/raf/DATASET
/content/raf/DATASET/test
/content/raf/DATASET/test/1
/content/raf/DATASET/test/2
/content/raf/DATASET/test/3
/content/raf/DATASET/test/4
/content/raf/DATASET/test/5
/content/raf/DATASET/test/6
/content/raf/DATASET/test/7
/content/raf/DATASET/train
/content/raf/DATASET/train/1
/content/raf/DATASET/train/2
/content/raf/DATASET/train/3
/content/raf/DATASET/train/4
/content/raf/DATASET/train/5
/content/raf/DATASET/train/6
/content/raf/DATASET/train/7


In [5]:
import os
base = "/content/raf/DATASET"
for split in ["train", "test"]:
    counts = {c: len(os.listdir(f"{base}/{split}/{c}")) for c in sorted(os.listdir(f"{base}/{split}"))}
    print(split, sum(counts.values()))

train 12271
test 3068


In [6]:
%%writefile train_mobilenet.py
import math, time, argparse, random, os, csv
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.utils.flop_counter import FlopCounterMode
from torchvision import datasets, transforms, models
from torchvision.models.mobilenetv2 import InvertedResidual as V2Block
from torchvision.models.mobilenetv3 import InvertedResidual as V3Block
from torchvision.ops.misc import SqueezeExcitation
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from tqdm import tqdm


class ECA(nn.Module):
    def __init__(self, c, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(c) + b) / gamma))
        k = t if t % 2 else t + 1
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        y = F.adaptive_avg_pool2d(x, 1).squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * torch.sigmoid(y)

class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 4)
        self.fc = nn.Sequential(nn.Linear(c, h), nn.ReLU(inplace=True), nn.Linear(h, c))
    def forward(self, x):
        y = self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))
        return x * torch.sigmoid(y)[:, :, None, None]

class CBAM(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 4)
        self.mlp = nn.Sequential(nn.Conv2d(c, h, 1, bias=False), nn.ReLU(inplace=True),
                                 nn.Conv2d(h, c, 1, bias=False))
        self.spatial = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        a = self.mlp(F.adaptive_avg_pool2d(x, 1))
        m = self.mlp(F.adaptive_max_pool2d(x, 1))
        x = x * torch.sigmoid(a + m)
        s = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)
        return x * torch.sigmoid(self.spatial(s))

ATTN = {"eca": ECA, "se": SE, "se_new": SE, "cbam": CBAM}


def build_model(backbone="mobilenetv3l", attn="se", num_classes=7, pretrained=True):
    if backbone == "mobilenetv2":

        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
        if attn != "none":
            for blk in m.features:
                if isinstance(blk, V2Block):
                    blk.conv = nn.Sequential(*blk.conv, ATTN[attn](blk.out_channels))
        m.classifier[1] = nn.Linear(m.last_channel, num_classes)

    elif backbone == "mobilenetv3l":

        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None)
        if attn != "se":
            n_replaced = 0
            for blk in m.features:
                if isinstance(blk, V3Block):
                    for i, layer in enumerate(blk.block):
                        if isinstance(layer, SqueezeExcitation):
                            c = layer.fc1.in_channels
                            blk.block[i] = nn.Identity() if attn == "none" else ATTN[attn](c)
                            n_replaced += 1
            print(f"Replaced {n_replaced} SE modules with '{attn}'")
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    else:
        raise ValueError(backbone)
    return m


MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_loaders(root, bs, workers, size):
    train_tf = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.9, 1.1)),
        transforms.ColorJitter(0.3, 0.3, 0.2),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((size, size)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

    full_train = datasets.ImageFolder(f"{root}/train", train_tf)
    full_eval = datasets.ImageFolder(f"{root}/train", eval_tf)
    test_set = datasets.ImageFolder(f"{root}/test", eval_tf)

    targets = np.array(full_train.targets)

    tr_idx, va_idx = train_test_split(np.arange(len(targets)), test_size=0.1,
                                      stratify=targets, random_state=42)
    counts = np.bincount(targets[tr_idx], minlength=len(full_train.classes))
    weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float)

    kw = dict(num_workers=workers, pin_memory=True)
    return (DataLoader(Subset(full_train, tr_idx), bs, shuffle=True, drop_last=True, **kw),
            DataLoader(Subset(full_eval, va_idx), bs, **kw),
            DataLoader(test_set, bs, **kw),
            full_train.classes, weights)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        ps.append(model(x.to(device)).argmax(1).cpu()); ys.append(y)
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return 100 * (y == p).mean(), 100 * f1_score(y, p, average="macro"), y, p

def fps_of(model, size, device, n=100):
    model = model.to(device).eval()
    x = torch.randn(1, 3, size, size, device=device)
    with torch.no_grad():
        for _ in range(20): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(n): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
    return n / (time.time() - t)

def efficiency(model, size, device):
    params = sum(p.numel() for p in model.parameters()) / 1e6
    cpu_model = model.to("cpu").eval()
    with FlopCounterMode(display=False) as fc:
        cpu_model(torch.randn(1, 3, size, size))
    gmacs = fc.get_total_flops() / 2 / 1e9
    fps_cpu = fps_of(cpu_model, size, torch.device("cpu"), n=30)
    fps_gpu = fps_of(model, size, device) if device.type == "cuda" else float("nan")
    return params, gmacs, fps_cpu, fps_gpu

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--backbone", default="mobilenetv3l", choices=["mobilenetv2", "mobilenetv3l"])
    ap.add_argument("--attn", default="se", choices=["none", "se", "se_new", "cbam", "eca"])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--size", type=int, default=112)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--out", default="best_mobile.pth")
    ap.add_argument("--log", default=None)
    a = ap.parse_args()
    if a.log is None:
        a.log = f"results_{a.backbone}.csv"

    seed_everything(a.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_dl, val_dl, test_dl, classes, cw = get_loaders(a.data, a.bs, a.workers, a.size)
    model = build_model(a.backbone, a.attn, len(classes)).to(device)

    crit = nn.CrossEntropyLoss(weight=cw.to(device), label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=a.epochs)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_f1 = -1.0
    for ep in range(1, a.epochs + 1):
        model.train(); loss_sum = 0.0
        for x, y in tqdm(train_dl, desc=f"Epoch {ep}/{a.epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            loss_sum += loss.item()
        sched.step()
        v_acc, v_f1, _, _ = evaluate(model, val_dl, device)
        print(f"Epoch {ep}: loss {loss_sum/len(train_dl):.4f} | val acc {v_acc:.2f} | val macro-F1 {v_f1:.2f}")
        if v_f1 > best_f1:
            best_f1 = v_f1; torch.save(model.state_dict(), a.out)

    model.load_state_dict(torch.load(a.out, map_location=device))
    t_acc, t_f1, y, p = evaluate(model, test_dl, device)
    cm = confusion_matrix(y, p)
    per_class = 100 * cm.diagonal() / cm.sum(1)
    print(f"\n[{a.backbone} | {a.attn} | seed {a.seed}] TEST acc {t_acc:.2f} | macro-F1 {t_f1:.2f}")
    print("Per-class acc:", {c: round(float(v), 1) for c, v in zip(classes, per_class)})
    print("Confusion matrix:\n", cm)

    params, gmacs, fps_cpu, fps_gpu = efficiency(model, a.size, device)
    print(f"Params {params:.3f}M | GMACs {gmacs:.3f} | FPS cpu {fps_cpu:.1f} | FPS gpu {fps_gpu:.1f}")

    new = not os.path.exists(a.log)
    with open(a.log, "a", newline="") as f:
        w = csv.writer(f)
        if new: w.writerow(["backbone", "attn", "seed", "size", "best_val_f1", "test_acc", "test_f1",
                            "params_M", "gmacs", "fps_cpu", "fps_gpu"])
        w.writerow([a.backbone, a.attn, a.seed, a.size, round(best_f1, 2), round(t_acc, 2), round(t_f1, 2),
                    round(params, 4), round(gmacs, 4), round(fps_cpu, 1), round(fps_gpu, 1)])

if __name__ == "__main__":
    main()

Writing train_mobilenet.py


In [7]:
!python train_mobilenet.py --data /content/raf/DATASET --backbone mobilenetv3l --attn eca --epochs 1 --out t.pth --log t.csv

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth
100% 21.1M/21.1M [00:00<00:00, 163MB/s]
Replaced 8 SE modules with 'eca'
Epoch 1: loss 1.9643 | val acc 41.78 | val macro-F1 30.25

[mobilenetv3l | eca | seed 0] TEST acc 43.61 | macro-F1 31.16
Per-class acc: {'1': 59.0, '2': 33.8, '3': 14.4, '4': 70.6, '5': 28.2, '6': 45.7, '7': 7.4}
Confusion matrix:
 [[194  41   9  52  18  11   4]
 [ 11  25   0  15   2  21   0]
 [ 27  15  23  52  22  14   7]
 [ 79  98  30 837  85  44  12]
 [ 66  50  39 136 135  46   6]
 [ 11  25   8  33  11  74   0]
 [176 102  83 146  90  33  50]]
Params 2.699M | GMACs 0.058 | FPS cpu 90.5 | FPS gpu 159.9


In [8]:
!for s in 0 1 2; do for a in se eca cbam; do python train_mobilenet.py --data /content/raf/DATASET --backbone mobilenetv3l --attn $a --seed $s --out mb3_${a}_s$s.pth && cp results_mobilenetv3l.csv train_mobilenet.py /content/drive/MyDrive/; done; done

Epoch 1: loss 1.8794 | val acc 57.82 | val macro-F1 49.43
Epoch 2: loss 1.5584 | val acc 62.46 | val macro-F1 54.83
Epoch 3: loss 1.4117 | val acc 69.46 | val macro-F1 61.16
Epoch 4: loss 1.3167 | val acc 72.07 | val macro-F1 63.43
Epoch 5: loss 1.2508 | val acc 74.02 | val macro-F1 65.09
Epoch 6: loss 1.1979 | val acc 73.53 | val macro-F1 66.96
Epoch 7: loss 1.1325 | val acc 66.21 | val macro-F1 59.66
Epoch 8: loss 1.0920 | val acc 73.21 | val macro-F1 65.20
Epoch 9: loss 1.0512 | val acc 74.59 | val macro-F1 65.91
Epoch 10: loss 1.0190 | val acc 76.47 | val macro-F1 67.46
Epoch 11: loss 0.9916 | val acc 73.62 | val macro-F1 66.63
Epoch 12: loss 0.9489 | val acc 74.43 | val macro-F1 66.57
Epoch 13: loss 0.9304 | val acc 76.30 | val macro-F1 68.95
Epoch 14: loss 0.9208 | val acc 77.20 | val macro-F1 69.50
Epoch 15: loss 0.9009 | val acc 78.91 | val macro-F1 70.86
Epoch 16: loss 0.8941 | val acc 78.42 | val macro-F1 69.68
Epoch 17: loss 0.8823 | val acc 76.38 | val macro-F1 69.45
Epoch 

In [9]:
import pandas as pd
df = pd.read_csv("results_mobilenetv3l.csv", keep_default_na=False)
g = df.groupby("attn").agg(
    n=("seed", "count"),
    acc_mean=("test_acc", "mean"), acc_std=("test_acc", "std"),
    f1_mean=("test_f1", "mean"), f1_std=("test_f1", "std"),
    params_M=("params_M", "first"), gmacs=("gmacs", "first"),
    fps_cpu=("fps_cpu", "mean"), fps_gpu=("fps_gpu", "mean")
).round(3)
print(g)
g.to_csv("summary_mobilenetv3l.csv")
!cp summary_mobilenetv3l.csv /content/drive/MyDrive/

      n  acc_mean  acc_std  f1_mean  f1_std  params_M  gmacs  fps_cpu  fps_gpu
attn                                                                          
cbam  3    80.207    0.327   71.017   0.370     3.076  0.059   68.700  121.733
eca   3    80.103    0.186   71.123   0.310     2.699  0.058   94.100  179.700
se    3    80.163    0.465   71.170   0.291     4.211  0.059   87.533  161.700


In [10]:
!ls -lh /content/*.pth

-rw-r--r-- 1 root root 12M Oct 31 08:33 /content/mb3_cbam_s0.pth
-rw-r--r-- 1 root root 12M Oct 31 09:21 /content/mb3_cbam_s1.pth
-rw-r--r-- 1 root root 12M Oct 31 10:09 /content/mb3_cbam_s2.pth
-rw-r--r-- 1 root root 11M Oct 31 08:17 /content/mb3_eca_s0.pth
-rw-r--r-- 1 root root 11M Oct 31 09:05 /content/mb3_eca_s1.pth
-rw-r--r-- 1 root root 11M Oct 31 09:53 /content/mb3_eca_s2.pth
-rw-r--r-- 1 root root 17M Oct 31 08:00 /content/mb3_se_s0.pth
-rw-r--r-- 1 root root 17M Oct 31 08:47 /content/mb3_se_s1.pth
-rw-r--r-- 1 root root 17M Oct 31 09:36 /content/mb3_se_s2.pth
-rw-r--r-- 1 root root 11M Oct 31 07:47 /content/t.pth


In [11]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/mb3_*.pth /content/results_mobilenetv3l.csv /content/summary_mobilenetv3l.csv /content/train_mobilenet.py /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive | grep -E "mb3|results_mobilenet|summary_mobilenet|train_mobilenet"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root  12M Oct 31 10:15 mb3_cbam_s0.pth
-rw------- 1 root root  12M Oct 31 10:15 mb3_cbam_s1.pth
-rw------- 1 root root  12M Oct 31 10:15 mb3_cbam_s2.pth
-rw------- 1 root root  11M Oct 31 10:15 mb3_eca_s0.pth
-rw------- 1 root root  11M Oct 31 10:15 mb3_eca_s1.pth
-rw------- 1 root root  11M Oct 31 10:15 mb3_eca_s2.pth
-rw------- 1 root root  17M Oct 31 10:15 mb3_se_s0.pth
-rw------- 1 root root  17M Oct 31 10:15 mb3_se_s1.pth
-rw------- 1 root root  17M Oct 31 10:15 mb3_se_s2.pth
-rw------- 1 root root  684 Oct 31 10:15 results_mobilenetv3l.csv
-rw------- 1 root root  236 Oct 31 10:15 summary_mobilenetv3l.csv
-rw------- 1 root root  11K Oct 31 10:15 train_mobilenet.py


In [12]:
!cp /content/train_fer.py /content/results_all.csv /content/summary_table.csv /content/drive/MyDrive/

cp: cannot stat '/content/train_fer.py': No such file or directory
cp: cannot stat '/content/results_all.csv': No such file or directory
cp: cannot stat '/content/summary_table.csv': No such file or directory


In [14]:
import pandas as pd
rows = [

 ["none", 0, 83.38, 75.70, 11.180, 0.485, 41.9, 320.1],
 ["none", 1, 83.41, 75.09, 11.180, 0.485, 41.1, 349.6],
 ["none", 2, 84.75, 77.39, 11.180, 0.485, 43.7, 350.9],
 ["se",   0, 83.08, 74.84, 11.269, 0.485, 38.0, 240.3],
 ["se",   1, 83.80, 76.64, 11.269, 0.485, 40.0, 231.6],
 ["se",   2, 84.19, 76.83, 11.269, 0.485, 39.8, 243.6],
 ["eca",  0, 83.08, 75.28, 11.180, 0.485, 37.8, 281.3],
 ["eca",  1, 83.74, 75.51, 11.180, 0.485, 39.8, 288.3],
 ["eca",  2, 83.77, 76.63, 11.180, 0.485, 39.4, 285.7],
 ["cbam", 0, 83.83, 76.39, 11.268, 0.485, 32.7, 162.3],
 ["cbam", 1, 84.13, 76.53, 11.268, 0.485, 36.0, 182.2],
 ["cbam", 2, 82.53, 73.68, 11.268, 0.485, 32.6, 170.8],
]
cols = ["attn","seed","test_acc","test_f1","params_M","gmacs","fps_cpu","fps_gpu"]
pd.DataFrame(rows, columns=cols).to_csv("results_resnet18.csv", index=False)
!cp results_resnet18.csv /content/drive/MyDrive/